# SOV Extraction — Approach 4: `xlsx` via preflighted PDF → 800 DPI TIFF

Approaches 1–3 in [`01_extract_sov.ipynb`](01_extract_sov.ipynb) dispatch by file shape:
- **Approach 1**: PDF + `method=extract` (grounding + per-field confidence).
- **Approach 2**: xlsx + `method=generate` (LLM, no grounding).
- **Approach 3**: xlsx with embedded images + per-image fan-out + client-side merge.

**Approach 4** (this notebook) gets the **best of both**: send xlsx through CU's PDF/image pipeline so we can use `method=extract` against it and recover grounding + confidence + bounding boxes for spreadsheet inputs.

## Pipeline

```
 .xlsx ─► (openpyxl) page-setup preflight ─► (LibreOffice) vector PDF ─► (pypdfium2 + Pillow) multi-page TIFF @ 800 DPI ─► CU sovExtractV1 (image/tiff)
```

Why each step:

- **Preflight** — force fit-to-1-page-wide, autofit column widths, expand the print area to cover anchored images. Without this, default Excel print rendering wraps wide schedules across many pages with mid-row column breaks.
- **PDF** — CU's *Standard (Layout)* pipeline accepts PDFs; xlsx itself only goes through the *Minimal* tier and can't ground `method=extract`. (See [research_xlsx_extract.ipynb](../../../feedback/underwriting/research_xlsx_extract.ipynb).)
- **TIFF @ 800 DPI** — `prebuilt-layout` reads vector PDFs' embedded text streams in a way that collapses adjacent borderless rows. Rasterizing forces the OCR path, which uses pure spatial geometry. 800 DPI eliminates the last OCR drift we saw at 600 DPI. (See [BUG_REPORT.md](../../../feedback/underwriting/research-output/pdfs/BUG_REPORT.md).)
- **Same analyzer** — we reuse the existing `sovExtractV1` template unchanged. The PDF-only Approach 1 and this Approach 4 share one analyzer; only the wire format differs.

## When to use which approach

| Input | Recommended | Why |
|---|---|---|
| PDF (native or scanned) | **Approach 1** | Native fit. |
| xlsx, no broker idiosyncrasies | **Approach 4** | Grounding + confidence. |
| xlsx, embedded images | **Approach 4** | Subsumes Pattern C — the rasterized pages include the image. |
| xlsx, you don't have LibreOffice installed | **Approach 2** (current production default) | No preprocessing dependency. |

## Prerequisites

Same as `01_extract_sov.ipynb`, plus:

- **LibreOffice** on PATH. Linux: `apt-get install -y libreoffice-calc fonts-dejavu`. Windows: `winget install TheDocumentFoundation.LibreOffice`.
- **Python deps**: `openpyxl`, `pypdfium2`, `Pillow`. All pure-Python wheels — Linux container friendly.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential

NB_DIR    = Path.cwd() if "__file__" not in globals() else Path(__file__).parent
DEMO_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR

# Make the preprocess library importable from the demo root.
if str(DEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMO_ROOT))

from preprocess import (
    apply_print_preflight,
    convert_libreoffice,
    find_libreoffice,
    rasterize_pdf_to_tiff,
)

ATTACH_DIR    = DEMO_ROOT / "attachments"
TEMPLATE_PATH = DEMO_ROOT / "reference" / "analyzer-templates" / "sov_extraction.json"
WORK_DIR      = DEMO_ROOT / ".cache" / "xlsx-via-pdf-tiff"   # intermediate artifacts
OUTPUT_DIR    = DEMO_ROOT / "reference" / "cu-output-tiff800"  # cached CU payloads for this approach
ENV_PATH      = DEMO_ROOT / ".env"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH)
load_dotenv(DEMO_ROOT.parent.parent / ".env")   # repo-root .env fallback

FORCE_REFRESH = False  # set True to re-run extraction even if cached results exist
TIFF_DPI = 800

ANALYZER_EXTRACT_ID = "sovExtractV1"  # reuses the same analyzer as Approach 1 (PDF)

print(f"demo root:    {DEMO_ROOT}")
print(f"attachments:  {ATTACH_DIR}  ({len(list(ATTACH_DIR.glob('*')))} files)")
print(f"work dir:     {WORK_DIR}")
print(f"output cache: {OUTPUT_DIR}")
print(f"libreoffice:  {find_libreoffice()}")
print(f"dpi:          {TIFF_DPI}")

## 2. CU client + analyzer template

We reuse the existing `sovExtractV1` analyzer template unchanged. The wire-format change is purely on the client side.

In [ ]:
_client: ContentUnderstandingClient | None = None

def get_client() -> ContentUnderstandingClient:
    global _client
    if _client is not None:
        return _client
    endpoint = (
        os.environ.get("CONTENTUNDERSTANDING_ENDPOINT")
        or os.environ.get("APP_CONTENT_UNDERSTANDING_ENDPOINT")
        or os.environ.get("FOUNDRY_ENDPOINT")
    )
    if not endpoint:
        raise RuntimeError("Set CONTENTUNDERSTANDING_ENDPOINT (or APP_CONTENT_UNDERSTANDING_ENDPOINT) in your .env.")
    _client = ContentUnderstandingClient(endpoint=endpoint, credential=DefaultAzureCredential())
    print(f"✅ client ready  endpoint={endpoint}")
    return _client

with open(TEMPLATE_PATH, "r", encoding="utf-8") as f:
    template_extract = json.load(f)

print(f"analyzer id:        {ANALYZER_EXTRACT_ID}")
print(f"baseAnalyzerId:     {template_extract['baseAnalyzerId']}")
print(f"# top-level fields: {len(template_extract['fieldSchema']['fields'])}")

## 3. Idempotent analyzer create

If `sovExtractV1` already exists (it does after running `01_extract_sov.ipynb`), this is a no-op.

In [ ]:
def ensure_analyzer(client: ContentUnderstandingClient, aid: str, body: dict) -> None:
    try:
        client.get_analyzer(analyzer_id=aid)
        print(f"↙️  reusing existing analyzer  {aid}")
        return
    except Exception:
        pass
    poller = client.begin_create_analyzer(analyzer_id=aid, resource=body)
    poller.result()
    print(f"✅ created analyzer  {aid}")

ensure_analyzer(get_client(), ANALYZER_EXTRACT_ID, template_extract)

## 4. Pipeline

Four steps wrapped into a single helper:

1. `apply_print_preflight(xlsx)` → preflighted xlsx in `WORK_DIR`.
2. `convert_libreoffice(preflighted)` → vector PDF.
3. `rasterize_pdf_to_tiff(pdf, dpi=800)` → multi-page TIFF.
4. `client.begin_analyze_binary(analyzer_id=sovExtractV1, content_type=image/tiff)` → CU payload.

Each step logs its elapsed seconds and the artifact path so reruns are easy to inspect.

In [ ]:
def _result_to_dict(r):
    if hasattr(r, "as_dict"):
        return r.as_dict()
    return json.loads(json.dumps(r, default=lambda o: getattr(o, "__dict__", str(o))))


def run_pipeline(xlsx_path: Path, *, dpi: int = TIFF_DPI) -> dict:
    """xlsx → preflight → PDF → TIFF@dpi → sovExtractV1. Returns the CU payload."""
    stem = xlsx_path.stem
    sample_work = WORK_DIR / stem
    sample_work.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    preflighted = apply_print_preflight(
        xlsx_path, sample_work / f"{stem}.print-ready.xlsx"
    )
    t_preflight = round(time.perf_counter() - t0, 2)
    print(f"  [1/4] preflight   -> {preflighted.relative_to(DEMO_ROOT)}  ({preflighted.stat().st_size:,} B)  {t_preflight}s")

    t0 = time.perf_counter()
    pdf = convert_libreoffice(preflighted, sample_work)
    t_pdf = round(time.perf_counter() - t0, 2)
    print(f"  [2/4] libreoffice -> {pdf.relative_to(DEMO_ROOT)}  ({pdf.stat().st_size:,} B)  {t_pdf}s")

    t0 = time.perf_counter()
    tiff = rasterize_pdf_to_tiff(pdf, sample_work, dpi=dpi)
    t_tiff = round(time.perf_counter() - t0, 2)
    print(f"  [3/4] rasterize   -> {tiff.relative_to(DEMO_ROOT)}  ({tiff.stat().st_size:,} B)  {t_tiff}s")

    client = get_client()
    t0 = time.perf_counter()
    with open(tiff, "rb") as f:
        poller = client.begin_analyze_binary(
            analyzer_id=ANALYZER_EXTRACT_ID,
            binary_input=f.read(),
            content_type="image/tiff",
        )
    payload = _result_to_dict(poller.result())
    t_cu = round(time.perf_counter() - t0, 2)
    print(f"  [4/4] cu analyze  -> {len((payload.get('contents') or [{}])[0].get('fields', {}))} top-level fields  {t_cu}s")

    payload.setdefault("_meta", {})
    payload["_meta"].update({
        "source_file": xlsx_path.name,
        "approach": "xlsx_via_pdf_tiff",
        "dpi": dpi,
        "timings": {
            "preflight_sec": t_preflight,
            "libreoffice_sec": t_pdf,
            "rasterize_sec": t_tiff,
            "cu_analyze_sec": t_cu,
            "total_sec": round(t_preflight + t_pdf + t_tiff + t_cu, 2),
        },
        "artifacts": {
            "preflighted_xlsx": str(preflighted.relative_to(DEMO_ROOT)),
            "pdf": str(pdf.relative_to(DEMO_ROOT)),
            "tiff": str(tiff.relative_to(DEMO_ROOT)),
        },
    })
    return payload

## 5. Run on every xlsx in `attachments/`

Results cached to `reference/cu-output-tiff800/<stem>.json`. The validator notebook ([`03_validate_extraction.ipynb`](03_validate_extraction.ipynb)) reads from this directory when comparing against ground truth.

PDFs in `attachments/` are skipped here — they belong to Approach 1 and are handled by `01_extract_sov.ipynb`.

In [ ]:
xlsx_files = sorted(ATTACH_DIR.glob("*.xlsx"))
print(f"found {len(xlsx_files)} xlsx samples in {ATTACH_DIR}")

results: dict[str, dict] = {}

for src in xlsx_files:
    cache_path = OUTPUT_DIR / f"{src.stem}.json"
    if cache_path.exists() and not FORCE_REFRESH:
        payload = json.loads(cache_path.read_text(encoding="utf-8"))
        payload.setdefault("_meta", {})["from_cache"] = True
        results[src.name] = payload
        print(f"\n=== {src.name}  (cached)")
        continue

    print(f"\n=== {src.name}")
    payload = run_pipeline(src)
    payload["_meta"]["from_cache"] = False
    cache_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    results[src.name] = payload

print(f"\n✅ {len(results)} samples processed; cached under {OUTPUT_DIR.relative_to(DEMO_ROOT)}")

## 6. Per-sample summary

Cell-population counts and mean grounded confidence. Compare against Approach 2's `cu-output/` cache via `03_validate_extraction.ipynb` to see the accuracy lift.

In [ ]:
import pandas as pd

def _summarize(payload: dict) -> dict:
    fields = (payload.get("contents") or [{}])[0].get("fields", {})
    scalars_t = scalars_p = 0
    for v in fields.values():
        if isinstance(v, dict) and v.get("type") in ("string", "number", "date", "integer"):
            scalars_t += 1
            if any(k.startswith("value") and v.get(k) not in (None, "", [], {}) for k in v):
                scalars_p += 1
    locs = (fields.get("Locations") or {}).get("valueArray", []) or []
    cells_t = cells_p = 0
    confs: list[float] = []
    for it in locs:
        for fv in (it.get("valueObject") or {}).values():
            cells_t += 1
            if any(k.startswith("value") and fv.get(k) not in (None, "", [], {}) for k in fv):
                cells_p += 1
                if isinstance(fv.get("confidence"), (int, float)):
                    confs.append(float(fv["confidence"]))
    return {
        "scalars": f"{scalars_p}/{scalars_t}",
        "locations_rows": len(locs),
        "location_cells": f"{cells_p}/{cells_t}",
        "cell_pct": round(cells_p * 100 / cells_t, 1) if cells_t else 0.0,
        "mean_conf": round(sum(confs) / len(confs), 3) if confs else None,
    }

rows = []
for name, payload in results.items():
    s = _summarize(payload)
    meta = payload.get("_meta", {})
    rows.append({
        "sample": name,
        **s,
        "total_sec": (meta.get("timings") or {}).get("total_sec"),
        "from_cache": meta.get("from_cache"),
    })
pd.DataFrame(rows)

## Next

Run [`03_validate_extraction.ipynb`](03_validate_extraction.ipynb) and toggle the cache directory to `cu-output-tiff800/` to see this approach's accuracy against ground truth.

Per the research notebook on Acme: this approach hits **100.0%** vs. ground truth at 800 DPI (vs. 99.0% at 600 DPI, 97.2% on the raw vector PDF, and 100.0% on the current `generate` route — but with the addition of grounded confidences and bounding boxes that `generate` doesn't provide).